# Olist 电商项目：从原始 CSV 到经营、分层与配送分析

依次运行各单元格。先将 9 个 CSV 放入 `data/raw/` 并安装项目依赖；图表、汇总 CSV 和分阶段文档会在项目目录内生成。结果仅用于历史观察，详细口径见 `docs/`。


In [ ]:
from pathlib import Path
import subprocess, sys
import pandas as pd
from IPython.display import display, Image

# 兼容从项目根目录或 notebooks 目录启动。
current = Path.cwd().resolve()
ROOT = next((p for p in [current, *current.parents] if (p / 'src/stage0.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('请从项目目录或 notebooks 目录启动 Jupyter')
missing = [name for name in ['olist_orders_dataset.csv', 'olist_order_items_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_customers_dataset.csv', 'olist_products_dataset.csv', 'olist_sellers_dataset.csv', 'olist_geolocation_dataset.csv', 'product_category_name_translation.csv'] if not (ROOT / 'data/raw' / name).is_file()]
if missing:
    raise FileNotFoundError(f'请先准备原始 CSV：{missing}')
print('项目目录：', ROOT)

def run(*args):
    subprocess.run([sys.executable, *args], cwd=ROOT, check=True)


## 阶段 0：数据理解与质量检查


In [ ]:
run('src/stage0.py')
display(pd.read_csv(ROOT/'results/tables/00_cleaning_log.csv'))
display(pd.read_csv(ROOT/'results/tables/00_foreign_keys.csv'))


## 阶段 1：经营指标及固定窗口复购


In [ ]:
run('src/stage1.py')
run('src/stage1_supplement.py')
display(pd.read_csv(ROOT/'results/tables/01_overall.csv'))
display(pd.read_csv(ROOT/'results/tables/01_repeat_90d.csv'))
display(Image(filename=str(ROOT/'results/figures/01_monthly_gmv.png')))


## 阶段 2：RFM 与六个运营人群


In [ ]:
run('src/stage2.py')
run('src/stage2_groups.py')
run('src/write_stage2_notes.py')
display(pd.read_csv(ROOT/'results/tables/02_frequency_distribution.csv'))
display(pd.read_csv(ROOT/'results/tables/02_operating_groups.csv'))
display(Image(filename=str(ROOT/'results/figures/02_operating_group_shares.png')))


## 阶段 3：自然日期主口径与时间戳敏感性对照


In [ ]:
run('src/stage3.py')
run('src/stage3.py','--policy','timestamp')
run('src/stage3_policy_comparison.py')
run('src/write_stage3_notes.py')
display(pd.read_csv(ROOT/'results/tables/03_policy_comparison.csv'))
display(Image(filename=str(ROOT/'results/figures/03_delay_bad_review.png')))


## 独立对账与合成边界测试


In [ ]:
run('validate.py')
run('-m','unittest','discover','-s','tests','-v')
print('运行结束；请结合 docs 中的数据覆盖和评价时间顺序限制解释结果。')
